In [1]:
# Standard libraries
import os
from pathlib import Path
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [2]:
path = 'results\compare'

# SET THIS MY DEFAULT PATH TO WHERE YOU HAVE THE FILES TO COMPARE
os.chdir(path)

df_80 = pd.read_csv('un_lolazo_submission.csv')
df_81 = pd.read_csv('rf_por_region.csv')
df_85 = pd.read_csv('cat_boost_perh.csv')

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\herie\AppData\Local\Temp\ipykernel_38056\1483177258.py:1: SyntaxWarning: invalid escape sequence '\c'
  path = 'results\compare'


In [3]:
import pandas as pd

# Cargar las predicciones
a = df_80
b = df_81
c = df_85
 
# Unir por id
df = a.merge(b, on="SamplingOperations_code", suffixes=("_81", "_80"))
df = df.merge(c, on="SamplingOperations_code")
df.rename(columns={"IBD_EQR_Status": "IBD_EQR_Status_85"}, inplace=True)


# Votación mayoritaria
df["consenso"] = df[["IBD_EQR_Status_81", "IBD_EQR_Status_80", "IBD_EQR_Status_85"]].mode(axis=1)[0]

# Coincidencia total (los 3 iguales)
df["coinciden_todos"] = (
    (df["IBD_EQR_Status_81"] == df["IBD_EQR_Status_80"]) & 
    (df["IBD_EQR_Status_81"] == df["IBD_EQR_Status_85"])
)

# Proporción de coincidencia
p_coincidencia = df["coinciden_todos"].mean()

print(f"Coincidencia total entre los 3 modelos: {p_coincidencia:.2%}")

# Guardar la predicción final (de consenso)
pred_coincidecnias = df[["SamplingOperations_code", "consenso"]]


Coincidencia total entre los 3 modelos: 79.30%


In [4]:
df

,SamplingOperations_code,IBD_EQR_Status_81,IBD_EQR_Status_80,IBD_EQR_Status_85,consenso,coinciden_todos
0,S02000010_20080811,Moderate,Good,Good,Good,False
1,S02000010_20100719,Good,Good,Good,Good,True
2,S02000010_20150811,Moderate,Good,Moderate,Moderate,False
3,S02000010_20170703,Good,Good,Good,Good,True
4,S02000011_20100719,Good,Moderate,Good,Good,False
...,...,...,...,...,...,...
5058,S06940940_20100708,Moderate,Moderate,Good,Moderate,False
5059,S06940940_20230623,Moderate,Good,Moderate,Moderate,False
5060,S06960950_20160629,High,High,High,High,True
5061,S06960950_20180719,High,High,High,High,True


In [5]:
pred_coincidecnias

,SamplingOperations_code,consenso
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Moderate
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [6]:
# Nombres de columnas (ajústalos si cambian)
col_id = "SamplingOperations_code"
col_a  = "IBD_EQR_Status_81"  # Modelo A (81%)
col_b  = "IBD_EQR_Status_80"  # Modelo B (80%)
col_c  = "IBD_EQR_Status_85"  # Modelo C (85%)

# =========================
# 1) Coincidencias par a par
# =========================
df["coincide_ab"] = df[col_a] == df[col_b]
df["coincide_ac"] = df[col_a] == df[col_c]
df["coincide_bc"] = df[col_b] == df[col_c]

# DFs filtrados SOLO con coincidencias (si prefieres ver solo los que sí coinciden)
df_ab = df.loc[df["coincide_ab"], [col_id, col_a, col_b]].copy()
df_ac = df.loc[df["coincide_ac"], [col_id, col_a, col_c]].copy()
df_bc = df.loc[df["coincide_bc"], [col_id, col_b, col_c]].copy()

# (opcional) también puedes quedarte con el flag en versión "completa":
# df_ab_full = df[[col_id, col_a, col_b, "coincide_ab"]].copy()
# df_ac_full = df[[col_id, col_a, col_c, "coincide_ac"]].copy()
# df_bc_full = df[[col_id, col_b, col_c, "coincide_bc"]].copy()

# =========================
# 2) Triple coincidencia
# =========================
df["coinciden_todos"] = (df[col_a] == df[col_b]) & (df[col_a] == df[col_c])
df_triple = df.loc[df["coinciden_todos"], [col_id, col_a, col_b, col_c]].copy()

# (métrica) proporción de triple coincidencia
p_triple = df["coinciden_todos"].mean()
print(f"Triple coincidencia (A=B=C): {p_triple:.2%}")

# =========================
# 3) Votación mayoritaria + % de coincidencia con A, B y C
# =========================
# Predicción de consenso por mayoría
df["consenso"] = df[[col_a, col_b, col_c]].mode(axis=1)[0]

# % de acuerdo del consenso con cada modelo
p_consenso_con_a = (df["consenso"] == df[col_a]).mean()
p_consenso_con_b = (df["consenso"] == df[col_b]).mean()
p_consenso_con_c = (df["consenso"] == df[col_c]).mean()

print(f"Consenso vs A (81%): {p_consenso_con_a:.2%}")
print(f"Consenso vs B (80%): {p_consenso_con_b:.2%}")
print(f"Consenso vs C (85%): {p_consenso_con_c:.2%}")

# DF final de consenso (id + predicción)
df_consenso = df[[col_id, "consenso"]].copy()

# =========================
# (Opcional) Guardados
# =========================
# df_ab.to_csv("results/coincidencias_ab.csv", index=False)
# df_ac.to_csv("results/coincidencias_ac.csv", index=False)
# df_bc.to_csv("results/coincidencias_bc.csv", index=False)
# df_triple.to_csv("results/triple_coincidencia.csv", index=False)
# df_consenso.to_csv("results/prediccion_consenso.csv", index=False)


Triple coincidencia (A=B=C): 79.30%
Consenso vs A (81%): 91.37%
Consenso vs B (80%): 94.71%
Consenso vs C (85%): 93.07%


In [7]:
df_triple

,SamplingOperations_code,IBD_EQR_Status_81,IBD_EQR_Status_80,IBD_EQR_Status_85
1,S02000010_20100719,Good,Good,Good
3,S02000010_20170703,Good,Good,Good
5,S02000011_20210823,Good,Good,Good
6,S02001016_20080811,Moderate,Moderate,Moderate
7,S02001016_20200707,Good,Good,Good
...,...,...,...,...
5055,S06840600_20190722,Moderate,Moderate,Moderate
5056,S06931250_20190627,Good,Good,Good
5060,S06960950_20160629,High,High,High
5061,S06960950_20180719,High,High,High


In [8]:
df_consenso

,SamplingOperations_code,consenso
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Moderate
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [9]:
from __future__ import annotations
import os
import math
import pandas as pd
from typing import List, Tuple, Dict, Optional

def evaluar_contra_modelos(
    df_nuevo: pd.DataFrame,
    modelos: List[Tuple[str, float]],
    *,
    id_col: str = "SamplingOperations_code",
    pred_col: str = "IBD_EQR_Status",
    carpeta: Optional[str] = None,  # si todos están en la misma carpeta, puedes pasarla aquí
    candidatos_pred_cols: Optional[List[str]] = None,  # override si ya conoces el nombre exacto
    normalizar_labels: bool = True,
) -> Dict[str, object]:
    """
    Compara las predicciones de `df_nuevo` contra varios modelos ya evaluados (guardados como CSV)
    y contra consensos (mayoría simple y mayoría ponderada por accuracies). También estima la
    probabilidad esperada de que la predicción de `df_nuevo` sea correcta.

    Parámetros
    ----------
    df_nuevo : DataFrame
        Debe contener [id_col, pred_col] con las predicciones a evaluar.
    modelos : List[Tuple[str, float]]
        Lista de (ruta_csv, accuracy) para cada modelo histórico evaluado.
        - 'accuracy' debe venir en [0,1]. Si la tienes en %, pásala/convierte a proporción.
    id_col : str
        Nombre de la columna id. Default: "SamplingOperations_code".
    pred_col : str
        Nombre de la columna de la predicción de df_nuevo. Default: "IBD_EQR_Status".
    carpeta : str | None
        Carpeta base donde están los CSV (si la ruta de cada CSV no es absoluta).
    candidatos_pred_cols : list[str] | None
        Lista de posibles nombres de columna de predicción dentro de cada CSV.
        Si None, se intenta autodetección inteligente.
    normalizar_labels : bool
        Si True, hace strip y upper() en labels string para robustez.

    Retorna
    -------
    dict con:
      - 'resumen_modelos': DataFrame (modelo, accuracy, n_overlap, coincidencia, archivo)
      - 'acuerdo_consenso_simple': float
      - 'acuerdo_consenso_ponderado': float
      - 'esperado_correcto_promedio': float
      - 'detallado': DataFrame con columnas:
            [id_col, pred_col, <preds de cada modelo>, consenso_simple, consenso_pond,
             prob_correcta_estimada, match_<modelo>...]
    """
    # --- Validaciones básicas
    if id_col not in df_nuevo.columns or pred_col not in df_nuevo.columns:
        raise ValueError(f"`df_nuevo` debe contener las columnas '{id_col}' y '{pred_col}'.")

    df_base = df_nuevo[[id_col, pred_col]].copy()

    # Normalización suave de labels para evitar mismatches por casing/espacios
    def _norm(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            s = x.strip()
            return s.upper() if normalizar_labels else s
        return x

    if normalizar_labels:
        df_base[pred_col] = df_base[pred_col].map(_norm)

    # Heurística de autodetección del nombre de columna de predicción
    default_candidates = [
        pred_col,
        "IBD_EQR_Status",
        "prediction",
        "pred",
        "status",
    ]
    # Permitimos columnas con prefijo, p.ej. IBD_EQR_Status_81
    def _find_pred_col(cols: List[str]) -> str:
        cands = (candidatos_pred_cols or default_candidates)
        # 1) coincidencia exacta por candidatos
        for c in cands:
            if c in cols:
                return c
        # 2) heurística: la que empiece con 'IBD_EQR_Status'
        for c in cols:
            if str(c).startswith("IBD_EQR_Status"):
                return c
        # 3) fallback: escoger la primera no-id con dtype 'object' o 'category'
        for c in cols:
            if c != id_col:
                return c
        raise ValueError("No pude detectar la columna de predicción del CSV del modelo.")

    # Cargar predicciones de modelos y armar tabla combinada
    tablas = []
    resumen_rows = []
    nombre_cols_modelo = []

    for ruta, acc in modelos:
        if acc > 1.0:  # si viene en %, conviértelo
            acc = acc / 100.0

        archivo = os.path.join(carpeta, ruta) if (carpeta and not os.path.isabs(ruta)) else ruta
        df_m = pd.read_csv(archivo)

        if id_col not in df_m.columns:
            raise ValueError(f"En '{archivo}' no existe la columna id '{id_col}'.")

        col_pred_m = _find_pred_col(df_m.columns.tolist())
        col_modelo = os.path.splitext(os.path.basename(archivo))[0]  # nombre corto desde el filename
        col_modelo_pred = f"pred__{col_modelo}"  # evitar colisiones

        tmp = df_m[[id_col, col_pred_m]].rename(columns={col_pred_m: col_modelo_pred})

        if normalizar_labels:
            tmp[col_modelo_pred] = tmp[col_modelo_pred].map(_norm)

        tablas.append(tmp)
        nombre_cols_modelo.append((col_modelo, col_modelo_pred, acc, archivo))

    # Merge incremental por id
    df_all = df_base.copy()
    for _, col_pred_m, _, _ in nombre_cols_modelo:
        # merge secuencial (left) para conservar todos los ids de df_nuevo
        df_all = df_all.merge(
            next(t for t in tablas if col_pred_m in t.columns),
            on=id_col, how="left"
        )

    # Coincidencia por modelo
    for col_modelo, col_pred_m, acc, archivo in nombre_cols_modelo:
        match_col = f"match__{col_modelo}"
        df_all[match_col] = (df_all[pred_col] == df_all[col_pred_m]) & df_all[pred_col].notna() & df_all[col_pred_m].notna()

        overlap = df_all[col_pred_m].notna().sum()
        coinc = df_all[match_col].mean() if overlap > 0 else float("nan")

        resumen_rows.append({
            "modelo": col_modelo,
            "accuracy_reportada": acc,
            "n_overlap": int(overlap),
            "coincidencia_con_df": float(coinc) if not math.isnan(coinc) else None,
            "archivo": archivo
        })

    resumen_modelos = pd.DataFrame(resumen_rows)

    # Consenso (mayoría simple: sin ponderar)
    cols_modelos_pred = [c for _, c, _, _ in nombre_cols_modelo]
    if len(cols_modelos_pred) == 0:
        raise ValueError("No se proporcionaron modelos para comparar.")

    def _mode_row(row_vals):
        # mode() fila a fila (ignorando NaNs). Si hay empate, devuelve el primero por orden.
        s = pd.Series(row_vals).dropna()
        if s.empty:
            return pd.NA
        return s.mode().iloc[0]

    df_all["consenso_simple"] = df_all[cols_modelos_pred].apply(_mode_row, axis=1)

    # Consenso ponderado por accuracies: etiqueta con suma de pesos (accuracies) más alta
    etiqueta_unica = set()
    # recolectar universo de etiquetas posibles (para robustez)
    for c in cols_modelos_pred:
        etiqueta_unica.update(df_all[c].dropna().unique().tolist())

    etiquetas = list(etiqueta_unica)

    pesos = {col_pred_m: acc for _, col_pred_m, acc, _ in nombre_cols_modelo}

    def _consenso_ponderado(row):
        # suma de accuracies por etiqueta propuesta
        best_label, best_w = (pd.NA, -1.0)
        for label in etiquetas:
            w = 0.0
            for col in cols_modelos_pred:
                val = row[col]
                if pd.isna(val): 
                    continue
                if val == label:
                    w += pesos[col]
            if w > best_w:
                best_label, best_w = label, w
        return best_label

    df_all["consenso_pond"] = df_all.apply(_consenso_ponderado, axis=1)

    # Acuerdo df vs consensos
    acuerdo_simple = (df_all[pred_col] == df_all["consenso_simple"]) & df_all[pred_col].notna() & df_all["consenso_simple"].notna()
    acuerdo_ponderado = (df_all[pred_col] == df_all["consenso_pond"]) & df_all[pred_col].notna() & df_all["consenso_pond"].notna()

    acuerdo_consenso_simple = acuerdo_simple.mean()
    acuerdo_consenso_ponderado = acuerdo_ponderado.mean()

    # Estimación de prob. de estar correcto por registro
    # Heurística: prob_correcta(id) = (suma de accuracies de modelos que coinciden con df_nuevo) / (suma de accuracies disponibles)
    sum_acc_total = sum(pesos.values()) if len(pesos) else 0.0

    def _prob_correcta_estimada(row):
        if pd.isna(row[pred_col]) or sum_acc_total == 0:
            return pd.NA
        agree_w = 0.0
        any_obs = False
        for col in cols_modelos_pred:
            val = row[col]
            if pd.isna(val): 
                continue
            any_obs = True
            if val == row[pred_col]:
                agree_w += pesos[col]
        if not any_obs:
            return pd.NA
        return agree_w / sum_acc_total

    df_all["prob_correcta_estimada"] = df_all.apply(_prob_correcta_estimada, axis=1)
    esperado_correcto_promedio = float(df_all["prob_correcta_estimada"].dropna().mean()) if df_all["prob_correcta_estimada"].notna().any() else None

    # Orden final de columnas “bonitas”
    orden_cols = [id_col, pred_col] + cols_modelos_pred + [f"match__{m}" for m, _, _, _ in nombre_cols_modelo] + ["consenso_simple", "consenso_pond", "prob_correcta_estimada"]
    detallado = df_all[orden_cols].copy()

    return {
        "resumen_modelos": resumen_modelos.sort_values("modelo").reset_index(drop=True),
        "acuerdo_consenso_simple": float(acuerdo_consenso_simple) if acuerdo_consenso_simple == acuerdo_consenso_simple else None,
        "acuerdo_consenso_ponderado": float(acuerdo_consenso_ponderado) if acuerdo_consenso_ponderado == acuerdo_consenso_ponderado else None,
        "esperado_correcto_promedio": esperado_correcto_promedio,
        "detallado": detallado
    }


In [10]:
df_nuevo = pd.read_csv('Boosted_ClassificationR.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo
print(salida["acuerdo_consenso_simple"])  # % acuerdo con mayoría simple
print(salida["acuerdo_consenso_ponderado"]) # % acuerdo con mayoría ponderada
print(salida["esperado_correcto_promedio"]) # estimado de acierto promedio de tu df
display(salida["detallado"])     # por-registro (incluye prob_correcta_estimada)


,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.847960,cat_boost_perh.csv
1,rf_por_region,0.8000,5663,0.842310,rf_por_region.csv
2,un_lolazo_submission,0.8100,5063,0.730532,un_lolazo_submission.csv


0.8552004238036377
0.8543174995585379
0.8074983125838665


,SamplingOperations_code,IBD_EQR_Status,pred__un_lolazo_submission,pred__rf_por_region,pred__cat_boost_perh,match__un_lolazo_submission,match__rf_por_region,match__cat_boost_perh,consenso_simple,consenso_pond,prob_correcta_estimada
0,S05169000_20130911,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
1,S05169000_20200819,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
2,S05172000_20210813,GOOD,GOOD,GOOD,GOOD,True,True,True,GOOD,GOOD,1.000000
3,S05172050_20210824,GOOD,POOR,GOOD,MODERATE,False,True,False,GOOD,MODERATE,0.324873
4,S05172350_20120827,HIGH,HIGH,HIGH,HIGH,True,True,True,HIGH,HIGH,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
5658,S03128500_20120723,MODERATE,MODERATE,MODERATE,MODERATE,True,True,True,MODERATE,MODERATE,1.000000
5659,S03128640_20130718,MODERATE,NaN,MODERATE,MODERATE,False,True,True,MODERATE,MODERATE,0.671066
5660,S03128640_20181011,MODERATE,GOOD,MODERATE,MODERATE,False,True,True,MODERATE,MODERATE,0.671066
5661,S03128732_20210817,MODERATE,MODERATE,MODERATE,GOOD,True,True,False,MODERATE,MODERATE,0.653807


In [11]:
df_nuevo = pd.read_csv('Catboost_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo


,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.853081,cat_boost_perh.csv
1,rf_por_region,0.8000,5663,0.846195,rf_por_region.csv
2,un_lolazo_submission,0.8100,5063,0.727883,un_lolazo_submission.csv


In [12]:
consenso_simple1 = salida["detallado"][["SamplingOperations_code", "consenso_simple"]]
consenso_simple1.to_csv("consenso_simple1.csv", index=False)



In [13]:
consenso_pond1 = salida["detallado"][["SamplingOperations_code", "consenso_pond"]]
consenso_pond1.to_csv("consenso_pond1.csv", index=False)

In [14]:
df_nuevo = pd.read_csv('Catboost_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

df_nuevo.to_csv("Catboost_Classificationr.csv", index=False)

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
    ("consenso_pond1.csv",1),
    ("consenso_simple1.csv",1)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.853081,cat_boost_perh.csv
1,consenso_pond1,1.0000,5663,0.856966,consenso_pond1.csv
2,consenso_simple1,1.0000,5663,0.857849,consenso_simple1.csv
3,rf_por_region,0.8000,5663,0.846195,rf_por_region.csv
4,un_lolazo_submission,0.8100,5063,0.727883,un_lolazo_submission.csv


In [15]:
df_nuevo = pd.read_csv('Boosted_Classificationr.csv')

df_nuevo = df_nuevo.rename(columns={"yhat_te": "IBD_EQR_Status"})

df_nuevo.to_csv("Boosted_Classificationr.csv", index=False)

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.80),
    ("cat_boost_perh.csv", 0.8525),
    ("consenso_pond1.csv",1),
    ("consenso_simple1.csv",1)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"])          # tabla con coincidencias por modelo

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,cat_boost_perh,0.8525,5663,0.847960,cat_boost_perh.csv
1,consenso_pond1,1.0000,5663,0.854317,consenso_pond1.csv
2,consenso_simple1,1.0000,5663,0.855200,consenso_simple1.csv
3,rf_por_region,0.8000,5663,0.842310,rf_por_region.csv
4,un_lolazo_submission,0.8100,5063,0.730532,un_lolazo_submission.csv


In [16]:
df_nuevo = pd.read_csv('cb_full_t.csv')



# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.831185,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.839131,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.897228,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.910295,cb_t.csv
4,rf_por_region,0.802489,5663,0.843899,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.766202,un_lolazo_submission.csv


0.849072957896299


In [17]:
df_nuevo = pd.read_csv('y_hat.csv')
df_nuevo = df_nuevo[["y_hat", "SamplingOperations_code"]]
df_nuevo = pd.DataFrame(df_nuevo)
df_nuevo = df_nuevo.rename(columns={"y_hat": "IBD_EQR_Status"})
df_nuevo

# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.815645,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.819354,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.851669,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.850079,cb_t.csv
4,rf_por_region,0.802489,5663,0.823592,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.720113,un_lolazo_submission.csv


0.8141846283202114


In [18]:
df_nuevo = pd.read_csv('cb_full.csv')



# Vector de tuplas (archivo_csv, accuracy_en_[0,1])
modelos = [
    ("un_lolazo_submission.csv", 0.81),
    ("rf_por_region.csv", 0.802489),
    ("cat_boost_perh.csv", 0.852459),
    ("Catboost_Classificationr.csv",.833695),
    ("Boosted_Classificationr.csv",.820462),
    ('cb_t.csv',0.868852)
]

salida = evaluar_contra_modelos(df_nuevo, modelos)
display(salida["resumen_modelos"]) 
print(salida['esperado_correcto_promedio'])

,modelo,accuracy_reportada,n_overlap,coincidencia_con_df,archivo
0,Boosted_Classificationr,0.820462,5663,0.826594,Boosted_Classificationr.csv
1,Catboost_Classificationr,0.833695,5663,0.835953,Catboost_Classificationr.csv
2,cat_boost_perh,0.852459,5663,0.896874,cat_boost_perh.csv
3,cb_t,0.868852,5663,0.895992,cb_t.csv
4,rf_por_region,0.802489,5663,0.850433,rf_por_region.csv
5,un_lolazo_submission,0.810000,5063,0.777856,un_lolazo_submission.csv


0.8481784013957122
